In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_parquet('../data/interim/fide_players_bruto.parquet')

In [19]:
print(df.info())
df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 1914517 entries, 0 to 1914516
Data columns (total 12 columns):
 #   Column        Dtype  
---  ------        -----  
 0   fideid        int64  
 1   name          str    
 2   country       str    
 3   sex           str    
 4   title         str    
 5   w_title       str    
 6   o_title       str    
 7   rating        int64  
 8   rapid_rating  int64  
 9   blitz_rating  int64  
 10  birthday      float64
 11  flag          str    
dtypes: float64(1), int64(4), str(7)
memory usage: 216.3 MB
None


,fideid,name,country,sex,title,w_title,o_title,rating,rapid_rating,blitz_rating,birthday,flag
0,10292519,"A A M Imtiaz, Chowdhury",BAN,M,NaN,NaN,NaN,0,0,0,1975.0,NaN
1,10688862,"A Abdel Maabod, Hoda",EGY,F,NaN,NaN,NaN,0,0,0,2009.0,w
2,577017641,A Adhisiva,IND,M,NaN,NaN,NaN,0,0,0,2015.0,NaN
3,577038320,A Akilesh Kumar,IND,M,NaN,NaN,NaN,0,0,0,2015.0,NaN
4,33496722,A Aman,IND,M,NaN,NaN,NaN,0,0,0,1996.0,NaN
5,577037219,A Anuj,IND,M,NaN,NaN,NaN,0,0,0,2015.0,NaN
6,537001345,A Arbhin Vanniarajan,IND,M,NaN,NaN,NaN,1450,1500,1431,2018.0,NaN
7,35853913,"A Aziz, Mohd Azizi Jamil",MAS,M,NaN,NaN,NaN,0,0,0,1981.0,NaN
8,35893303,"A Aziz, Mohd Khalis",MAS,M,NaN,NaN,NaN,0,0,0,1991.0,NaN
9,10224084,"A B M Hasibuzzaman, Tapan",BAN,M,NaN,NaN,NaN,0,0,0,1977.0,NaN


In [20]:

df['birthday'] = df['birthday'].astype('Int64')


In [21]:
df_limpio = df[
    (df['rating'] > 0) | 
    (df['rapid_rating'] > 0) | 
    (df['blitz_rating'] > 0)
]

df_limpio.info()


<class 'pandas.DataFrame'>
Index: 785987 entries, 6 to 1914511
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   fideid        785987 non-null  int64
 1   name          785987 non-null  str  
 2   country       785987 non-null  str  
 3   sex           785987 non-null  str  
 4   title         24668 non-null   str  
 5   w_title       5122 non-null    str  
 6   o_title       12867 non-null   str  
 7   rating        785987 non-null  int64
 8   rapid_rating  785987 non-null  int64
 9   blitz_rating  785987 non-null  int64
 10  birthday      776189 non-null  Int64
 11  flag          388605 non-null  str  
dtypes: Int64(1), int64(4), str(7)
memory usage: 95.9 MB


In [49]:

df_limpio = df_limpio.dropna(subset=['birthday'])

duplicados_id = df_limpio.duplicated(subset=['fideid']).sum()
print(f"Cantidad de registros con FIDE ID repetido: {duplicados_id}")

if duplicados_id > 0:
    filas_repetidas = df_limpio[df_limpio.duplicated(subset=['fideid'], keep=False)]

filas_repetidas



Cantidad de registros con FIDE ID repetido: 0


,fideid,name,country,sex,title,w_title,o_title,rating,rapid_rating,blitz_rating,birthday,flag
1645645,5019168,Sridharan Ramanathan,IND,M,NaN,NaN,NaN,1841,1702,1782,1965,i
1645646,5019168,Sridharan Ramanathan,IND,M,NaN,NaN,NaN,1841,1702,1782,1965,i


In [50]:
df_limpio = df_limpio.drop_duplicates(subset=['fideid'], keep='first')
df_limpio = df_limpio.drop(columns=['k','games','rapid_games', 'blitz_games'], errors='ignore')

df_limpio.info()

<class 'pandas.DataFrame'>
Index: 776188 entries, 6 to 1914511
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   fideid        776188 non-null  int64   
 1   name          776188 non-null  str     
 2   country       776188 non-null  category
 3   sex           776188 non-null  category
 4   title         776188 non-null  category
 5   w_title       776188 non-null  category
 6   o_title       776188 non-null  category
 7   rating        776188 non-null  int64   
 8   rapid_rating  776188 non-null  int64   
 9   blitz_rating  776188 non-null  int64   
 10  birthday      776188 non-null  Int64   
 11  flag          776188 non-null  category
dtypes: Int64(1), category(6), int64(4), str(1)
memory usage: 61.1 MB


In [51]:

columnas_titulos = ['title', 'w_title', 'o_title']
for c in columnas_titulos:
    df_limpio[c] = df_limpio[c].fillna('nt')
    df_limpio[c] = df_limpio[c].replace(['NaN'], 'nt')
    df_limpio[c] = df_limpio[c].replace(r'^\s*$', 'nt', regex=True)



df_limpio['flag'] = df_limpio['flag'].fillna('a')
df_limpio['flag'] = df_limpio['flag'].replace(['NaN'], 'a')
df_limpio['flag'] = df_limpio['flag'].replace(r'^\s*$', 'a', regex=True)

df_limpio.head()

,fideid,name,country,sex,title,w_title,o_title,rating,rapid_rating,blitz_rating,birthday,flag
6,537001345,A Arbhin Vanniarajan,IND,M,nt,nt,nt,1450,1500,1431,2018,a
10,10245154,"A B M Jobair, Hossain",BAN,M,nt,nt,nt,1715,1738,1748,1998,a
15,558074015,A Bala Phani Prabhanjan,IND,M,nt,nt,nt,1585,1553,1524,2012,a
17,12457558,A Bu Ba Co,VIE,M,nt,nt,nt,0,1638,0,2002,a
18,25121731,A C J John,IND,M,nt,nt,nt,1438,0,0,1987,i


In [52]:
columnas_categoricas = ['sex', 'country', 'flag', 'title', 'w_title', 'o_title']
for col in columnas_categoricas:
    df_limpio[col] = df_limpio[col].astype('category')

df_limpio.info()


<class 'pandas.DataFrame'>
Index: 776188 entries, 6 to 1914511
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   fideid        776188 non-null  int64   
 1   name          776188 non-null  str     
 2   country       776188 non-null  category
 3   sex           776188 non-null  category
 4   title         776188 non-null  category
 5   w_title       776188 non-null  category
 6   o_title       776188 non-null  category
 7   rating        776188 non-null  int64   
 8   rapid_rating  776188 non-null  int64   
 9   blitz_rating  776188 non-null  int64   
 10  birthday      776188 non-null  Int64   
 11  flag          776188 non-null  category
dtypes: Int64(1), category(6), int64(4), str(1)
memory usage: 61.1 MB


In [53]:
df_limpio.to_parquet('../data/processed/fide_players.parquet', index=False)